In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.interpolate import interp1d

In [ ]:
#### INPUT file paths to GITT data - paste just after the r, before the comma ####
sample_files = {
    "sample-1": [
        r,
    ],
    "sample-2": [
        r,
    ],
    "sample-3": [
        r,
    ],
}

#### INPUT mean particle radius of CAM material ####
rp = 0.0002  # mean CAM particle radius in cm

#### INPUT appropriate Arbin protocol step numbers ####
pulse_step = 7
rest_step = 8

#### plotting setup ####
fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True)
colors = plt.cm.tab20.colors

summary_data = []

#### process each sample ####
for i, (sample_name, file_list) in enumerate(sample_files.items()):
    color = colors[i % len(colors)]

    valid_voltage_ranges = []
    interp_diff = []

    for file_path in file_list:

        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"{file_path} : Failed to load - {e}")
            continue

        D_Li = []
        file_label = file_path.split('\\')[-1]

        GITT_steps = list(range(3, df['Cycle Index'].max() - 1))

        for step in GITT_steps:

            #### Charge analysis ####
            pulse_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == pulse_step)].copy()
            rest_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == rest_step)].copy()
            prev_rest_data = df[(df['Cycle Index'] == step - 1) & (df['Step Index'] == rest_step)].copy()

            if not (pulse_data.empty or rest_data.empty or prev_rest_data.empty):
                try:
                    tau = pulse_data['Test Time (s)'].iloc[-1] - pulse_data['Test Time (s)'].iloc[0]
                    t0 = pulse_data['Test Time (s)'].iloc[0]

                    # early pulse region (E ~ a + b*sqrt(t))
                    mask = (pulse_data['Test Time (s)'] - t0 >= 2) & \
                           (pulse_data['Test Time (s)'] - t0 <= 60)

                    dEt_data = pulse_data[mask]

                    if not dEt_data.empty:
                        sqrt_t = np.sqrt(dEt_data['Test Time (s)'] - t0)
                        voltage = dEt_data['Voltage (V)']

                        slope, _, _, _, _ = linregress(sqrt_t, voltage)
                        dEt = abs(slope)

                        dEs = abs(rest_data['Voltage (V)'].iloc[-1] -
                                  prev_rest_data['Voltage (V)'].iloc[-1])

                        Eeq_vals = rest_data['Voltage (V)'].iloc[-1]

                        if dEt != 0 and tau != 0:
                            D = (4 / (np.pi * 9)) * ((rp / tau) ** 2) * ((dEs / dEt) ** 2)

                            D_Li.append({
                                'Step': step,
                                'Diff_coef (cm^2/s)': D,
                                'Eeq (V)': Eeq_vals
                            })

                except Exception as e:
                    print(f"{file_label}: Error at step {step} - {e}")

        results_df = pd.DataFrame(D_Li)

        #### Raw scatter plot ####
        if not results_df.empty:

            axes[0].scatter(results_df['Eeq (V)'],
                            results_df['Diff_coef (cm^2/s)'],
                            label=f"{sample_name}",
                            color=color,
                            s=30,
                            edgecolors='black',
                            alpha=0.6)

            sorted_df = results_df.sort_values('Eeq (V)')
            vmin = sorted_df['Eeq (V)'].min()
            vmax = sorted_df['Eeq (V)'].max()

            valid_voltage_ranges.append((vmin, vmax))
            interp_diff.append((sorted_df['Eeq (V)'],
                                sorted_df['Diff_coef (cm^2/s)']))

    #### voltage grid for interpolation ####
    if valid_voltage_ranges:

        vmin_all = max(v[0] for v in valid_voltage_ranges)
        vmax_all = min(v[1] for v in valid_voltage_ranges)

        voltage_grid = np.linspace(vmin_all, vmax_all, 1000)

        interp_matrix = []
        for v_vals, d_vals in interp_diff:
            interp_func = interp1d(v_vals, d_vals,
                                   kind='linear',
                                   bounds_error=False,
                                   fill_value=np.nan)
            interp_matrix.append(interp_func(voltage_grid))

        diff_matrix = np.array(interp_matrix)

        mean_diff = np.nanmean(diff_matrix, axis=0)
        std_diff = np.nanstd(diff_matrix, axis=0)

        #### Plot mean ± std ####
        axes[1].plot(voltage_grid,
                     mean_diff,
                     color=color,
                     linewidth=2.5,
                     label=f"{sample_name} Mean")

        axes[1].fill_between(voltage_grid,
                             mean_diff - std_diff,
                             mean_diff + std_diff,
                             color=color,
                             alpha=0.3)

        #### Get peak diffusion coefficient ####
        max_idx = np.nanargmax(mean_diff)
        max_voltage = voltage_grid[max_idx]
        max_diff_val = mean_diff[max_idx]
        max_std_val = std_diff[max_idx]

        summary_data.append({
            "Sample": sample_name,
            "Voltage (V)": round(max_voltage, 3),
            "Mean Diff Coef (cm^2/s)": max_diff_val,
            "Std Dev": max_std_val,
            "Diff Coef (cm^2/s)": f"{max_diff_val:.2e} ± {max_std_val:.2e}"
        })

#### plot formatting ####
axes[0].set_title('GITT calculated Li diff coefficients', fontsize=18, fontweight='bold')
axes[0].set_ylabel('Li Diff Coef (cm$^2$/s)', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cell Potential (V)')
axes[0].set_ylim(-0.1e-11, 2.0e-11)
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].set_title("Mean and Standard Deviation", fontsize=18, fontweight='bold')
axes[1].set_ylabel('Li Diff Coef (cm$^2$/s)', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cell Potential (V)', fontsize=16, fontweight='bold')
axes[1].set_ylim(-0.1e-11, 2.0e-11)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=15)

plt.tight_layout()
fig.subplots_adjust(top=0.9)
fig.canvas.draw()

### INPUT name of study to save the figure ###
plt.savefig('study-1', dpi=300)
plt.show()

#### convert to dataframe ####
summary_df = pd.DataFrame(summary_data)

print("\n=== Peak Diffusion Summary ===\n")
print(summary_df[['Sample', 'Voltage (V)', 'Diff Coef (cm^2/s)']])

### INPUT name of study to save summary ###
summary_df.to_csv("diffusion_summary_study-1.csv", index=False)